# Lesson 06a Walkthrough: Why Good Database Design Matters
## Primary Keys, Foreign Keys & Redundancy

**Course:** Database Applications Development  
**Medina County Career Center**

---

In this walkthrough, we're going to look under the hood of the IMDb database you explored yesterday. We'll answer the question: **Why is this database split into 5 separate tables instead of 1 giant table?**

By the end, you'll understand three key concepts:
1. **Primary Keys** — how each row gets a unique ID
2. **Foreign Keys** — how tables link together
3. **Redundancy** — why storing duplicate data is a problem

In [ ]:
# Setup — same as yesterday
import pandas as pd
import sqlite3

dbPath = 'imdb_class_2010plus.db'
connection = sqlite3.connect(dbPath)
cursor = connection.cursor()

print(f'Connected to: {dbPath}')

---

## Part 1: What Are Primary Keys?

A **primary key** is a column (or set of columns) that uniquely identifies each row in a table. No two rows can have the same primary key value.

Think of it like your student ID — no two students share the same number, and it never changes. That's exactly what a primary key does for a database row.

Let's see the primary keys in our IMDb database.

### Exploring `title_basics`

The `tconst` column is the primary key for titles. Each movie or TV series gets a unique ID like `tt1375666` (that's Inception).

Let's prove it's unique — if it's truly a primary key, no value should appear more than once.

In [ ]:
# How many total rows vs how many UNIQUE tconst values?
# If these numbers match, every tconst is unique → it works as a primary key

query = '''
    SELECT 
        COUNT(*) as totalRows,
        COUNT(DISTINCT tconst) as uniqueTconst
    FROM title_basics
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))
print()

# Check: are they equal?
total = df['totalRows'][0]
unique = df['uniqueTconst'][0]
if total == unique:
    print(f'Every tconst is unique! {total:,} rows, {unique:,} unique IDs.')
    print('tconst works as a primary key.')
else:
    print(f'Problem! {total:,} rows but only {unique:,} unique values.')

### Why Not Use the Movie Title as the Primary Key?

You might wonder — why use a weird code like `tt1375666` instead of just using the movie's title? Let's see why.

In [ ]:
# Are movie titles unique? Let's find titles that appear more than once.

query = '''
    SELECT primaryTitle, COUNT(*) as count
    FROM title_basics
    GROUP BY primaryTitle
    HAVING count > 1
    ORDER BY count DESC
    LIMIT 15
'''

df = pd.read_sql_query(query, connection)
print('Titles that appear more than once:')
print(df.to_string(index=False))

There are multiple movies and TV series called "Home" or "Alone" or "The Batman"! If we used the title as the primary key, we'd have no way to tell them apart.

That's why we use `tconst` — each one is guaranteed unique, even when titles repeat.

### Exploring `name_basics`

The `nconst` column is the primary key for people. Each actor, director, or writer gets a unique ID.

In [ ]:
# Prove nconst is unique in name_basics

query = '''
    SELECT 
        COUNT(*) as totalRows,
        COUNT(DISTINCT nconst) as uniqueNconst
    FROM name_basics
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))
print()

total = df['totalRows'][0]
unique = df['uniqueNconst'][0]
if total == unique:
    print(f'Every nconst is unique! {total:,} rows, {unique:,} unique IDs.')
    print('nconst works as a primary key.')

In [ ]:
# And just like titles — are actor NAMES unique?

query = '''
    SELECT primaryName, COUNT(*) as count
    FROM name_basics
    GROUP BY primaryName
    HAVING count > 1
    ORDER BY count DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print('Names that appear more than once:')
print(df.to_string(index=False))

Multiple people named "David Smith" or "Michael Johnson" — names are NOT unique enough to be primary keys. `nconst` solves this.

### Primary Key Summary

| Table | Primary Key | Example |
|-------|------------|----------|
| title_basics | `tconst` | tt1375666 (Inception) |
| title_ratings | `tconst` | tt1375666 |
| name_basics | `nconst` | nm0000138 (Leonardo DiCaprio) |
| title_principals | `tconst` + `ordering` | tt1375666, 1 |
| title_crew_person | `tconst` + `nconst` + `role` | tt1375666, nm0634240, director |

Notice that `title_principals` and `title_crew_person` use **composite keys** — two or more columns together form the unique identifier. That's because one movie has many actors, and one actor is in many movies.

---

## Part 2: What Are Foreign Keys?

A **foreign key** is a column in one table that **references** the primary key of another table. It's the link that connects tables together.

In our database:
- `title_principals.tconst` is a **foreign key** pointing to `title_basics.tconst`
- `title_principals.nconst` is a **foreign key** pointing to `name_basics.nconst`

The foreign key says: "Go look up this ID in that other table to get the full details."

In [ ]:
# Let's see how foreign keys work in practice.
# title_principals has tconst and nconst — but just as IDs.
# We JOIN to get the actual names.

# First, raw data from title_principals (just IDs):
query = '''
    SELECT tconst, nconst, category
    FROM title_principals
    WHERE tconst = 'tt1375666'
    ORDER BY ordering
    LIMIT 5
'''

df = pd.read_sql_query(query, connection)
print('Raw title_principals data (just IDs):')
print(df.to_string(index=False))

In [ ]:
# Now JOIN to get the actual title and actor names
# The foreign keys let us "follow the links" to other tables

query = '''
    SELECT b.primaryTitle,
           n.primaryName,
           p.category
    FROM title_principals p
    JOIN title_basics b ON p.tconst = b.tconst
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE p.tconst = 'tt1375666'
    ORDER BY p.ordering
    LIMIT 5
'''

df = pd.read_sql_query(query, connection)
print('Same data, but with names (using JOINs on foreign keys):')
print(df.to_string(index=False))

See the difference? The raw data just has codes like `nm0000138`. By following the foreign key to `name_basics`, we find out that's Leonardo DiCaprio.

**The foreign key is the link that makes this possible.**

### Proving Foreign Keys Match

A foreign key should always point to a valid primary key in the other table. Let's verify — does every `tconst` in `title_principals` actually exist in `title_basics`?

In [ ]:
# Check: are there any tconst values in title_principals
# that DON'T exist in title_basics?
# If the foreign key is working correctly, this should return 0.

query = '''
    SELECT COUNT(*) as orphanedRows
    FROM title_principals p
    WHERE p.tconst NOT IN (SELECT tconst FROM title_basics)
'''

df = pd.read_sql_query(query, connection)
orphans = df['orphanedRows'][0]
print(f'Credits pointing to non-existent titles: {orphans}')

if orphans == 0:
    print('Every foreign key points to a valid title. The link is solid!')
else:
    print(f'Warning: {orphans} broken links found!')

### Foreign Key Map

Here's how all the tables connect:

```
title_basics  ←──(tconst)──  title_ratings
    ↑                              
  (tconst)                         
    │                              
title_principals  ──(nconst)──→  name_basics
    │                              ↑
    │                            (nconst)
    │                              │
title_crew_person ─────────────────┘
    │
  (tconst)──→ title_basics
```

The foreign keys are the arrows. Each one says "this ID column points to that table's primary key."

---

## Part 3: Why Redundancy Is the Enemy

Now for the big payoff — let's see what would happen if we **didn't** split the data into separate tables.

Imagine cramming everything into one big table where every credit row also stores the full movie title, year, genre, and the actor's name, birth year, and profession. That's redundancy — storing the same fact over and over.

In [ ]:
# How many credits does Inception have?

query = '''
    SELECT b.primaryTitle, COUNT(*) as creditCount
    FROM title_principals p
    JOIN title_basics b ON p.tconst = b.tconst
    WHERE b.primaryTitle = 'Inception'
      AND b.startYear = 2010
    GROUP BY b.primaryTitle
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))
print()
print('In a bad design, "Inception" and all its info would be')
print(f'stored {df["creditCount"][0]} times instead of once!')

In [ ]:
# Let's calculate the TOTAL redundancy across the whole database.
# For every title, how many times would the title info be duplicated?

query = '''
    SELECT 
        COUNT(*) as totalCredits,
        COUNT(DISTINCT tconst) as uniqueTitles
    FROM title_principals
'''

df = pd.read_sql_query(query, connection)
totalCredits = df['totalCredits'][0]
uniqueTitles = df['uniqueTitles'][0]

print(f'Total credits in title_principals: {totalCredits:,}')
print(f'Unique titles: {uniqueTitles:,}')
print(f'\nIn a BAD design (one big table):')
print(f'  Title info would be stored {totalCredits:,} times')
print(f'\nIn our GOOD design (separate tables):')
print(f'  Title info is stored {uniqueTitles:,} times (once each)')
print(f'\nWe save {totalCredits - uniqueTitles:,} duplicate copies!')

### The Update Problem

Redundancy isn't just about wasted space — it makes updates dangerous.

In [ ]:
# How many rows would we need to update if a movie title changes?
# Let's check a few popular movies.

query = '''
    SELECT b.primaryTitle, COUNT(*) as rowsToUpdate
    FROM title_principals p
    JOIN title_basics b ON p.tconst = b.tconst
    WHERE b.primaryTitle IN (
        'Inception', 'Interstellar', 'The Batman', 
        'Avengers: Endgame', 'Spider-Man: No Way Home'
    )
    GROUP BY b.primaryTitle
    ORDER BY rowsToUpdate DESC
'''

df = pd.read_sql_query(query, connection)
print('If title info was stored in every row (BAD design):')
print('Changing one movie title would require updating this many rows:')
print()
print(df.to_string(index=False))
print()
print('In our GOOD design: just 1 row each in title_basics!')

### The Typo Problem

When data is stored multiple times, typos can creep in. Let's simulate what that looks like.

In [ ]:
# In a BAD design, the same movie might be spelled differently:

print('Imagine a one-big-table design where someone misspells a title...')
print()
print('Row 1: Spider-Man: No Way Home    | Tom Holland  | actor')
print('Row 2: Spider-Man: No Way Home    | Zendaya      | actress')
print('Row 3: Spiderman: No Way Home     | Benedict C.  | actor')   # typo!
print('Row 4: Spider-Man: No Way Home    | Jacob Batalon| actor')
print()
print('Row 3 has a typo! Now a search for "Spider-Man: No Way Home"')
print('would miss Benedict Cumberbatch\'s credit.')
print()
print('In our GOOD design: the title is stored ONCE in title_basics.')
print('Credits just reference the ID. No chance of a title typo!')

---

## Part 4: Seeing It All Together

Let's trace a complete example through all the tables to see how primary keys, foreign keys, and no-redundancy work together.

In [ ]:
# Step 1: Find a movie in title_basics (using PK: tconst)
query = '''
    SELECT tconst, primaryTitle, startYear, genres
    FROM title_basics
    WHERE primaryTitle = 'The Dark Knight Rises'
'''
df = pd.read_sql_query(query, connection)
print('Step 1 — title_basics (PK = tconst):')
print(df.to_string(index=False))

movieId = df['tconst'][0]
print(f'\nThe movie ID is: {movieId}')

In [ ]:
# Step 2: Get rating from title_ratings (FK: tconst → title_basics)
query = f'''
    SELECT tconst, averageRating, numVotes
    FROM title_ratings
    WHERE tconst = '{movieId}'
'''
df = pd.read_sql_query(query, connection)
print('Step 2 — title_ratings (FK tconst links to title_basics):')
print(df.to_string(index=False))

In [ ]:
# Step 3: Get the cast from title_principals (FK: tconst → title_basics)
query = f'''
    SELECT tconst, nconst, category, characters
    FROM title_principals
    WHERE tconst = '{movieId}'
      AND category IN ('actor', 'actress', 'director')
    ORDER BY ordering
'''
df = pd.read_sql_query(query, connection)
print('Step 3 — title_principals (FK tconst, FK nconst):')
print(df.to_string(index=False))

In [ ]:
# Step 4: Follow the nconst FK to get actual names from name_basics
query = f'''
    SELECT n.nconst, n.primaryName, n.birthYear, p.category, p.characters
    FROM title_principals p
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE p.tconst = '{movieId}'
      AND p.category IN ('actor', 'actress', 'director')
    ORDER BY p.ordering
'''
df = pd.read_sql_query(query, connection)
print('Step 4 — Follow FK nconst → name_basics to get real names:')
print(df.to_string(index=False))
print()
print('Each step followed a foreign key to get more information.')
print('No data was duplicated — each fact is stored exactly ONCE.')

---

## Recap: The Three Rules

**1. Primary Keys** — Every table has a unique identifier
- `title_basics` → `tconst`
- `name_basics` → `nconst`  
- Names and titles are NOT good primary keys (they can duplicate)

**2. Foreign Keys** — Tables connect through ID references
- `title_principals.tconst` → points to `title_basics.tconst`
- `title_principals.nconst` → points to `name_basics.nconst`
- JOINs use foreign keys to pull data from multiple tables

**3. No Redundancy** — Each fact stored once
- "Inception" appears once in `title_basics`, not 19 times
- "Leonardo DiCaprio" appears once in `name_basics`, not 45 times
- We save hundreds of thousands of duplicate copies

**Now it's your turn — open the task notebook!**

In [ ]:
# Clean up
connection.close()
print('Database connection closed.')